# Inspect results and native lineage

Use an existing training or inference execution. The notebook reads its actual jobs and outputs. Native graph publication is optional and does not run model code.


## Choose the execution

- Locate this checkout and import the inspection helpers.
- Paste the full SageMaker pipeline execution ARN.
- Use the notebook's normal AWS identity in the execution's Region.

**Effect:** prepares the client and configuration.


In [ ]:
from pathlib import Path
import sys
import json
import boto3
import pandas as pd

root = Path.cwd().resolve()
while not (root / "mlops-models/src").is_dir() and root != root.parent:
    root = root.parent
sys.path[:0] = [str(root / "mlops-models/src"), str(root / "mlops-models")]
from mlops_models.pipelines.tracking import collect_execution, index_execution
from mlops_models.pipelines.lineage import publish_lineage, query_lineage

execution_arn = ""  # Paste one existing execution ARN.
region = execution_arn.split(":")[3] if execution_arn else "us-east-1"
session = boto3.Session(region_name=region)


## Inspect jobs and write the run index

- Read the selected execution, its step statuses and output references.
- Show which producer supplied each step, including cache reuse.
- Save the navigation index beside this execution's outputs.

**Effect:** reads native/S3 records and writes the small S3 index.

**Where to look:** open the printed `run.json` S3 location for artifact, job, Registry and available MLflow links.


In [ ]:
if not execution_arn:
    raise ValueError("Set execution_arn in the configuration cell first.")
report = collect_execution(session, execution_arn)
display(pd.DataFrame([
    {"step": step["name"], "status": step["status"], "cached": step["cached"],
     "producer": step["producer_execution_arn"], "job": step.get("job_arn", "")}
    for step in report["steps"]
]))
print("Run index:", index_execution(session, execution_arn))


## Publish the optional native graph

- Set `publish_native` to true when ready to create native lineage records.
- Link data/source locations and models to SageMaker's existing job records; no duplicate job records are created.
- Reuse existing records on repeat calls and save their references to S3.
- Keep model code, data files and approval status unchanged.

**Effect:** creates native lineage metadata and its receipt only when enabled.

**Where to look:** the printed `lineage.json` contains native artifact/job ARNs and relationships.


In [ ]:
publish_native = False
native = None
if publish_native:
    native = publish_lineage(session, execution_arn)
    print("Native lineage receipt:", native["receipt_uri"])
    print("Run index:", index_execution(session, execution_arn))
    display(pd.DataFrame([{"location": uri, "artifact": arn}
                          for uri, arn in native["artifacts"].items()]))
else:
    print("Native publication is off. Set publish_native=True to try it.")


## Query the native relationships

- Start from the registered model artifact, or choose another returned artifact/job ARN.
- Read the upstream relationships from SageMaker's lineage service.
- Display the actual returned edges for inspection.

**Effect:** reads native lineage only. The service follows up to ten relationship levels per query; start from an intermediate job to explore further upstream.

**Where to look:** the source/destination ARNs correspond to the native records in the preceding cell and saved receipt.


In [ ]:
start_arn = native["package_roots"][0] if native and native["package_roots"] else ""
if start_arn:
    graph = query_lineage(session, start_arn)
    display(pd.DataFrame(graph["Edges"]))
else:
    print("Publish the graph above, or set start_arn to an existing native artifact/job ARN.")
